# Task 2: Weighted Data Cleaning and Preprocessing

This notebook prepares the 2025 SHED survey for analysis while preserving survey design. It retains respondents with structural skips, preserves survey weights, reports missingness and outliers, and excludes identifiers, weights, and imputation flags from model features.

Blank survey cells are represented as `__NOT_ASKED__` in the modeling copy because skip logic is different from ordinary missingness. The raw dataframe is never overwritten.

## 1. Imports and configuration

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

DATA_PATH = Path('../data/public2025.csv')
EXPECTED_ROWS = 12_934
WEIGHT_COLUMNS = ['weight', 'weight_pop', 'panel_weight', 'panel_weight_pop']
ID_COLUMNS = ['shedid']
NOT_ASKED = '__NOT_ASKED__'

## 2. Load and validate the raw survey

`weight_pop` is used for population estimates. The other survey weights remain available for alternate analyses.

In [19]:
raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Raw shape: {raw.shape[0]:,} rows x {raw.shape[1]:,} columns')
required = {'shedid', *WEIGHT_COLUMNS}
missing_required = required.difference(raw.columns)
assert not missing_required, f'Missing required columns: {sorted(missing_required)}'
assert raw['shedid'].notna().all(), 'Respondent IDs contain blanks'
assert raw['shedid'].is_unique, 'Respondent IDs are duplicated'
assert raw[['weight', 'weight_pop']].notna().all().all(), 'Primary survey weights must be present'
weight_values = raw[WEIGHT_COLUMNS]
assert ((weight_values >= 0) | weight_values.isna()).all().all(), 'Present survey weights must be non-negative'
assert np.isfinite(weight_values.dropna().to_numpy(dtype=float)).all(), 'Present survey weights must be finite'
if len(raw) != EXPECTED_ROWS:
    print(f'Note: expected {EXPECTED_ROWS:,} rows from the project brief; found {len(raw):,}.')
else:
    print('Row count matches the project brief.')

Raw shape: 12,934 rows x 815 columns
Row count matches the project brief.


## 3. Profile missingness and imputation flags

In [20]:
iflag_columns = [column for column in raw.columns if column.endswith('_iflag')]
profile = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'blank_count': raw.isna().sum(),
    'blank_rate': raw.isna().mean(),
}).sort_values('blank_rate', ascending=False)
print(f'Imputation flag columns: {len(iflag_columns):,}')
display(profile.head(20))

Imputation flag columns: 357


,dtype,blank_count,blank_rate
BK49B,float64,12823,0.991418
A8_d,str,12818,0.991031
A8_c,str,12812,0.990567
A8_e,str,12808,0.990258
S18,str,12794,0.989176
R5C_c,str,12773,0.987552
R5C_b,str,12773,0.987552
R5C_d,str,12773,0.987552
R5C_a,str,12773,0.987552
S21,str,12655,0.978429


## 4. Preserve skip logic

Blank cells are not a reason to drop respondents. Existing response labels such as declined or don't know remain unchanged. The codebook should be consulted before recoding categories.

In [21]:
def mark_structural_skips(frame, columns=None, marker=NOT_ASKED):
    result = frame.copy()
    if columns is None:
        protected = set(ID_COLUMNS + WEIGHT_COLUMNS)
        columns = [column for column in result.columns if column not in protected]
    for column in columns:
        result[column] = result[column].astype('object').where(result[column].notna(), marker)
    return result

clean = mark_structural_skips(raw)
assert len(clean) == len(raw), 'Cleaning unexpectedly removed respondents'
assert clean['shedid'].equals(raw['shedid']), 'Respondent IDs changed during cleaning'
assert clean[WEIGHT_COLUMNS].equals(raw[WEIGHT_COLUMNS]), 'Survey weights changed during cleaning'
print(f'Rows retained: {len(clean):,} of {len(raw):,}')
print(f'Blank cells remaining in modeling copy: {int(clean.isna().sum().sum()):,}')

Rows retained: 12,934 of 12,934
Blank cells remaining in modeling copy: 17,030


## 5. Select features without leakage

Imputation flags describe imputation and must not predict the corresponding survey answers. Protected demographic variables remain available for later subgroup auditing. State geography is excluded from this general-purpose feature matrix.

In [22]:
def feature_columns(frame, iflags):
    excluded = set(ID_COLUMNS + WEIGHT_COLUMNS + iflags)
    excluded.update(column for column in frame.columns if column.lower() in {'ppstaten', 'state'})
    return [column for column in frame.columns if column not in excluded]

feature_names = feature_columns(clean, iflag_columns)
X = clean[feature_names].copy()
assert not set(X.columns).intersection(iflag_columns), 'Imputation flags leaked into features'
assert not set(X.columns).intersection(ID_COLUMNS + WEIGHT_COLUMNS), 'IDs or weights leaked into features'
print(f'Candidate features: {X.shape[1]:,}')

Candidate features: 452


## 6. Report numeric outliers

The IQR rule is a screening diagnostic, not proof that a survey response is invalid. No values are changed here; capping or transformation requires a codebook-based decision.

In [23]:
def iqr_outlier_report(frame):
    rows = []
    for column in frame.select_dtypes(include=np.number).columns:
        values = frame[column].dropna()
        if values.empty:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = int(((values < lower) | (values > upper)).sum())
        rows.append({'column': column, 'q1': q1, 'q3': q3, 'lower_bound': lower, 'upper_bound': upper, 'outlier_count': count, 'outlier_rate': count / len(values)})
    return pd.DataFrame(rows).sort_values('outlier_count', ascending=False)

outlier_report = iqr_outlier_report(raw.drop(columns=WEIGHT_COLUMNS, errors='ignore'))
display(outlier_report.head(20))

,column,q1,q3,lower_bound,upper_bound,outlier_count,outlier_rate
14,ppcmdate,20250207.0,2.025031e+07,2.025005e+07,2.025046e+07,4382,0.338797
23,ppc2date,20250130.0,2.025032e+07,2.024985e+07,2.025060e+07,3411,0.320372
16,ppp2date,20250604.0,2.025073e+07,2.025042e+07,2.025091e+07,3060,0.260714
18,pph1date,20250412.0,2.025061e+07,2.025012e+07,2.025090e+07,2567,0.225988
1,duration,953.0,1.963000e+03,-5.620000e+02,3.478000e+03,1772,0.137003
0,shedid,202300108.5,2.025036e+08,2.019948e+08,2.028089e+08,1457,0.112649
10,pphhsize,2.0,3.000000e+00,5.000000e-01,4.500000e+00,1438,0.111180
26,pphhsize5,2.0,3.000000e+00,5.000000e-01,4.500000e+00,1438,0.111180
20,ppmpdate,20250428.0,2.025051e+07,2.025030e+07,2.025064e+07,1139,0.096878
130,GH14_iflag,0.0,0.000000e+00,0.000000e+00,0.000000e+00,971,0.075073


## 7. Weighted summaries

In [24]:
def effective_sample_size(weights):
    weights = pd.Series(weights, dtype='float64').dropna()
    return float(weights.sum() ** 2 / weights.pow(2).sum())

def weighted_distribution(frame, column, weight_column='weight_pop'):
    values = frame[[column, weight_column]].dropna(subset=[weight_column]).copy()
    summary = values.groupby(column, dropna=False)[weight_column].agg(['sum', 'count']).rename(columns={'sum': 'weighted_count', 'count': 'unweighted_count'})
    summary['weighted_proportion'] = summary['weighted_count'] / summary['weighted_count'].sum()
    return summary.sort_values('weighted_proportion', ascending=False)

print(f"Population-weighted total: {raw['weight_pop'].sum():,.2f}")
print(f"Effective sample size: {effective_sample_size(raw['weight_pop']):,.1f}")
for column in ['EF1', 'B2']:
    if column in raw.columns:
        print(f'Weighted distribution for {column}:')
        display(weighted_distribution(raw, column))

Population-weighted total: 264,519,125.02
Effective sample size: 11,307.0
Weighted distribution for EF1:


,weighted_count,unweighted_count,weighted_proportion
EF1,,,
Yes,1.448642e+08,7327,0.547651
No,1.196550e+08,5607,0.452349


Weighted distribution for B2:


,weighted_count,unweighted_count,weighted_proportion
B2,,,
Doing okay,1.030964e+08,5005,0.389750
Living comfortably,8.980862e+07,4479,0.339517
Just getting by,4.952366e+07,2387,0.187221
Finding it difficult to get by,2.209042e+07,1063,0.083512


## 8. Build a preprocessing-ready matrix

Fit this transformer only on the training split when modeling to prevent preprocessing leakage.

In [25]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_string', FunctionTransformer(lambda values: values.astype(str))),
    ('one_hot', encoder),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
], remainder='drop')
X_encoded = preprocessor.fit_transform(X)
assert X_encoded.shape[0] == len(raw), 'Encoding removed respondents'
print(f'Encoded matrix shape: {X_encoded.shape[0]:,} rows x {X_encoded.shape[1]:,} columns')

Encoded matrix shape: 12,934 rows x 14,306 columns


## 9. Cleaning report and final checks

In [26]:
cleaning_report = pd.Series({
    'raw_rows': len(raw),
    'raw_columns': raw.shape[1],
    'rows_retained': len(clean),
    'rows_dropped': len(raw) - len(clean),
    'iflag_columns': len(iflag_columns),
    'candidate_features': len(feature_names),
    'numeric_features': len(numeric_features),
    'categorical_features': len(categorical_features),
    'encoded_features': X_encoded.shape[1],
    'population_weight_sum': raw['weight_pop'].sum(),
    'effective_sample_size': effective_sample_size(raw['weight_pop']),
})
display(cleaning_report.to_frame('value'))
assert cleaning_report['rows_dropped'] == 0
assert not set(feature_names).intersection(iflag_columns)
print('Task 2 preprocessing checks passed.')

,value
raw_rows,1.293400e+04
raw_columns,8.150000e+02
rows_retained,1.293400e+04
rows_dropped,0.000000e+00
iflag_columns,3.570000e+02
candidate_features,4.520000e+02
numeric_features,0.000000e+00
categorical_features,4.520000e+02
encoded_features,1.430600e+04
population_weight_sum,2.645191e+08


Task 2 preprocessing checks passed.


## 10. Export the cleaned respondent-level data

The original source file remains unchanged. This export contains the cleaned survey responses with structural skips marked as `__NOT_ASKED__`, while respondent IDs and survey weights are preserved.

In [27]:
CLEAN_OUTPUT_PATH = Path('../data/public2025_clean.csv')
clean.to_csv(CLEAN_OUTPUT_PATH, index=False)

assert CLEAN_OUTPUT_PATH.exists(), 'Clean CSV was not created'
exported = pd.read_csv(CLEAN_OUTPUT_PATH, low_memory=False)
assert exported.shape == clean.shape, 'Exported CSV shape does not match clean dataframe'
assert exported['shedid'].equals(clean['shedid']), 'Exported respondent IDs do not match clean dataframe'
print(f'Wrote {exported.shape[0]:,} rows x {exported.shape[1]:,} columns to {CLEAN_OUTPUT_PATH}')

Wrote 12,934 rows x 815 columns to ..\data\public2025_clean.csv
